# Dwipada Gemma 3 4B — QLoRA Fine-tuning

**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Make sure your HuggingFace account has Gemma 3 4B access approved
3. Add two Colab Secrets (left sidebar → 🔑):
   - `HF_TOKEN` = a HuggingFace token with **WRITE** scope
   - `HF_REPO`  = your target repo name e.g. `your-username/dwipada-gemma3-4b`
4. Upload `ift_alpaca.jsonl` when Cell 6 prompts you

**What this notebook does:**
- Installs Unsloth (fastest QLoRA trainer, native Gemma 3 support)
- Loads Gemma 3 4B in 4-bit — fits in ~9GB VRAM leaving headroom
- Trains with LoRA rank 32 (conservative for T4)
- Orders data: synthetic first → human after (simulates curriculum in one pass)
- Pushes LoRA adapter (~150MB) directly to HuggingFace Hub after training
- Checkpoints also pushed to Hub every 500 steps — safe against Colab disconnects

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Verify T4 is available — should show ~15109 MiB
# If you see 'No devices found', go to Runtime → Change runtime type → GPU

In [ ]:
# ── Cell 3: HuggingFace Login & Repo Config ───────────────────
# Token needs WRITE scope to push the adapter after training.
# Create one at: https://huggingface.co/settings/tokens
#   → New token → Role: Write
#
# Store both values as Colab Secrets (left sidebar → key icon):
#   HF_TOKEN = hf_xxxxxxxxxxxx
#   HF_REPO  = your-username/dwipada-gemma3-4b

from huggingface_hub import login, create_repo
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    HF_REPO  = userdata.get('HF_REPO')   # e.g. 'samvaran/dwipada-gemma3-4b'
except Exception:
    raise RuntimeError(
        "Add HF_TOKEN and HF_REPO to Colab Secrets (left sidebar → key icon)"
    )

login(token=HF_TOKEN, add_to_git_credential=False)

# Create the repo if it doesn't exist yet — safe to run even if it already exists
create_repo(HF_REPO, repo_type='model', exist_ok=True, private=True)

print(f"✅ Logged in. Adapter will be pushed to: https://huggingface.co/{HF_REPO}")
print(f"   Repo is PRIVATE — change private=True to False if you want it public.")

In [ ]:
# ── Cell 3: HuggingFace Login ──────────────────────────────────
# You need a HF token with READ access to google/gemma-3-4b-it
# Get it from: https://huggingface.co/settings/tokens
from huggingface_hub import login
from google.colab import userdata

# Option A (recommended): Store token as a Colab Secret
#   Left sidebar → Key icon → Add secret: HF_TOKEN = hf_xxxx
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in via Colab secret.")
except Exception:
    # Option B: paste token directly (less secure)
    login()  # will prompt interactively

In [ ]:
# ── Cell 4: Load Gemma 3 4B in 4-bit ──────────────────────────
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024   # Conservative for T4 — poems + meanings fit easily
                         # Increase to 2048 if you upgrade to Pro/A100

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name       = "google/gemma-3-4b-it",
    max_seq_length   = MAX_SEQ_LENGTH,
    dtype            = None,          # auto-detects bfloat16 on T4
    load_in_4bit     = True,          # QLoRA — keeps VRAM under 10GB
)

print(f"Model loaded. Dtype: {model.dtype}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 5: Attach LoRA Adapters ──────────────────────────────
# rank=32 is conservative for T4 — gives good capacity without OOM
# target_modules covers attention + FFN for full stylistic learning

model = FastLanguageModel.get_peft_model(
    model,
    r                    = 32,
    lora_alpha           = 64,        # 2x rank — standard stable ratio
    lora_dropout         = 0.05,
    target_modules       = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # FFN
    ],
    bias                 = "none",
    use_gradient_checkpointing = "unsloth",   # saves ~4GB VRAM on T4
    random_state         = 42,
    use_rslora           = False,
)

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable/1e6:.1f}M / {total/1e6:.0f}M "
      f"({100*trainable/total:.2f}%)")

In [ ]:
# ── Cell 6: Upload & Prepare Dataset ──────────────────────────
# Upload ift_alpaca.jsonl using the cell below OR mount Drive

# Option A: Direct upload
from google.colab import files
print("Upload ift_alpaca.jsonl when the dialog appears...")
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DATASET_PATH}")

# Option B: Mount Drive (comment out Option A and uncomment below)
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/dwipada/ift_alpaca.jsonl'

In [ ]:
# ── Cell 7: Order Data (Curriculum Simulation) ─────────────────
# Synthetic records first → human records after
# This mimics curriculum learning within a single training pass

import json
from datasets import Dataset

def load_alpaca_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

raw = load_alpaca_jsonl(DATASET_PATH)

# Split and reorder: synthetic first, human after
synthetic = [r for r in raw if r.get('_source_tag') == '[Synthetic]']
human     = [r for r in raw if r.get('_source_tag') == '[Human_Style]']
ordered   = synthetic + human

print(f"Total records : {len(ordered)}")
print(f"Synthetic     : {len(synthetic)} (first in training order)")
print(f"Human         : {len(human)} (second in training order)")

# Convert to HuggingFace Dataset
dataset = Dataset.from_list([
    {"instruction": r["instruction"], "output": r["output"]}
    for r in ordered
])
print(f"\nDataset ready: {dataset}")

In [ ]:
# ── Cell 8: Apply Alpaca Prompt Template ──────────────────────
# Gemma 3 IT uses a chat template internally, but Unsloth's
# alpaca formatter wraps it correctly for instruction fine-tuning

ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def format_prompts(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instr, out in zip(instructions, outputs):
        text = ALPACA_PROMPT.format(instr, out) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

# Quick sanity check — print one example
print("── Sample formatted record ──────────────────")
print(dataset[0]["text"][:600])
print("...")

In [ ]:
# ── Cell 9: Configure Trainer ─────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# T4-safe hyperparameters
# per_device_train_batch_size=1 + gradient_accumulation=8
# → effective batch size of 8 without OOM

trainer = SFTTrainer(
    model        = model,
    tokenizer    = tokenizer,
    train_dataset= dataset,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = True,      # packs multiple short poems per sequence
                                    # critical for throughput on small poems
    args = TrainingArguments(
        per_device_train_batch_size  = 1,
        gradient_accumulation_steps  = 8,    # effective batch = 8
        warmup_steps                 = 100,
        num_train_epochs             = 2,    # 2 epochs good for 29k records on T4
                                             # increase to 3 if loss hasn't converged
        learning_rate                = 2e-4,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 50,
        optim                        = 'adamw_8bit',   # 8-bit Adam saves ~2GB VRAM
        weight_decay                 = 0.01,
        lr_scheduler_type            = 'cosine',
        seed                         = 42,
        output_dir                   = './dwipada_checkpoints',
        save_strategy                = 'steps',
        save_steps                   = 500,
        save_total_limit             = 2,
        # ── HuggingFace Hub checkpointing ──────────────────────
        push_to_hub                  = True,
        hub_model_id                 = HF_REPO,
        hub_token                    = HF_TOKEN,
        hub_strategy                 = 'checkpoint',  # pushes every save_steps
                                                       # so Colab disconnects
                                                       # can't kill your progress
        report_to                    = 'none',
    ),
)

print('Trainer configured.')
print(f'Effective batch size : {1 * 8}')
print(f'Max sequence length  : {MAX_SEQ_LENGTH}')
print(f'Epochs               : 2')
print(f'Checkpoints → Hub    : {HF_REPO} (every 500 steps)')

In [ ]:
# ── Cell 10: VRAM Check Before Training ───────────────────────
# Should show < 12GB used — leaving ~3GB headroom for training
import torch
gpu_stats   = torch.cuda.get_device_properties(0)
used_vram   = torch.cuda.memory_allocated() / 1e9
total_vram  = gpu_stats.total_memory / 1e9
print(f"GPU           : {gpu_stats.name}")
print(f"VRAM used     : {used_vram:.2f} GB")
print(f"VRAM total    : {total_vram:.2f} GB")
print(f"VRAM free     : {total_vram - used_vram:.2f} GB")

if used_vram > 12:
    print("\n⚠️  WARNING: VRAM usage is high. Consider reducing MAX_SEQ_LENGTH to 512.")
else:
    print("\n✅ VRAM looks good — safe to start training.")

In [ ]:
# ── Cell 11: TRAIN ────────────────────────────────────────────
# Expected time on T4:
#   ~5-7 hours for 2 epochs over 29k records with packing=True
# Keep the Colab tab active or use a browser extension to prevent timeout

trainer_stats = trainer.train()

print(f"\nTraining complete.")
print(f"Total steps     : {trainer_stats.global_step}")
print(f"Training loss   : {trainer_stats.training_loss:.4f}")
print(f"Runtime         : {trainer_stats.metrics['train_runtime']/3600:.2f} hours")

In [ ]:
# ── Cell 12: Quick Inference Test ────────────────────────────
# Run this before saving to verify the model actually learned something

FastLanguageModel.for_inference(model)   # enables 2x faster inference

TEST_POEM = "భువనత్రయాధారభూతమయుండు \nపవనుండు లేకున్న బడు శరీరములు"

test_prompt = ALPACA_PROMPT.format(
    f"""[Human_Style] [Minimalist]
Assume role of a Telugu and Sanskrit scholar and give me bhavam and prathipadartham of the following dwipada poem. If there are combined words please break them with + in prathipadartham. Further bhavam should be in single line in telugu and English. Just give only bhavam and prathipadartham of the given input. No additional data.\nPoem:\n\n{TEST_POEM}""",
    ""   # empty — model fills this in
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens  = 512,
    temperature     = 0.7,
    top_p           = 0.9,
    do_sample       = True,
    pad_token_id    = tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Strip the prompt — show only the generated response
response_only = response[len(test_prompt):].strip()

print("── Model Response ───────────────────────────")
print(response_only)

In [ ]:
# ── Cell 13: Push LoRA Adapter to HuggingFace Hub ────────────
# Pushes only the adapter weights (~150MB), not the full 8GB model.
# The adapter is usable by anyone who loads google/gemma-3-4b-it as base.

SAVE_DIR = './dwipada_lora_adapter'

# Save locally first (fast)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Adapter saved locally to {SAVE_DIR}')

# Push to Hub
print(f'Pushing to https://huggingface.co/{HF_REPO} ...')
model.push_to_hub(
    HF_REPO,
    token        = HF_TOKEN,
    commit_message = 'Add Dwipada QLoRA adapter — final',
)
tokenizer.push_to_hub(
    HF_REPO,
    token        = HF_TOKEN,
    commit_message = 'Add tokenizer',
)

print(f'\n✅ Adapter pushed successfully.')
print(f'   View at: https://huggingface.co/{HF_REPO}')

# List local files for reference
import os
print('\nLocal adapter files:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1e6
    print(f'  {f:<45} {size:.1f} MB')

In [ ]:
# ── Cell 14: Verify Push & Print Reload Code ─────────────────
# Confirms the adapter is on Hub and prints the exact code
# you need to reload it anywhere.

from huggingface_hub import list_repo_files

print(f'Files in {HF_REPO}:')
for f in list_repo_files(HF_REPO, token=HF_TOKEN):
    print(f'  {f}')

print(f'''
── How to reload this adapter anywhere ─────────────────

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "{HF_REPO}",
    max_seq_length = 1024,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
─────────────────────────────────────────────────────────
''')

## Reloading the Adapter from HuggingFace

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "your-username/dwipada-gemma3-4b",  # your HF_REPO value
    max_seq_length = 1024,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
```

## If Colab Disconnects Mid-Training

Checkpoints are pushed to HuggingFace Hub every 500 steps automatically.
To resume, re-run all setup cells then point `output_dir` to the last
checkpoint folder name shown in your Hub repo — Unsloth picks up from there.

## Key Numbers to Watch

| Metric | Good | Concerning |
|---|---|---|
| Training loss after epoch 1 | < 1.0 | > 2.0 |
| VRAM during training | < 14GB | > 14.5GB (risk of OOM) |
| Steps/second | ~1.5–2.5 | < 0.5 (something wrong) |
| Hub checkpoint visible | After step 500 | Not visible after 30min = check hub_token |